In [ ]:
!pip install selenium webdriver-manager

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import time
import os

def check_lazada_html(url, save_file="lazada_html.html"):
    chrome_options = Options()
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("--disable-infobars")
    chrome_options.add_argument("--disable-notifications")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

    print("Đang khởi động trình duyệt ChromeDriver...")
    
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)

    try:
        print(f"Đang truy cập: {url}")
        driver.get(url)
        
        print("Đang đợi trang load 10 giây (bạn hãy giải captcha nếu có)...")
        time.sleep(10) # Tăng thời gian chờ lên một xíu để bạn có thời gian kéo captcha nếu bị hỏi
        
        # Cuộn trang từ từ để nạp hết các sản phẩm và ảnh
        for i in range(5):
            driver.execute_script(f"window.scrollTo(0, {(i+1)*800});")
            time.sleep(1)
        
        html_source = driver.page_source
        
        # LƯU FILE HTML
        with open(save_file, "w", encoding="utf-8") as f:
            f.write(html_source)
            
        print("-" * 50)
        print(f"✅ THÀNH CÔNG: Đã lưu mã HTML vào file {os.path.abspath(save_file)}!")
        print(f"Độ dài mã HTML: {len(html_source)} ký tự.")
        print("-" * 50)
        
        return html_source

    except Exception as e:
        print(f"❌ Xảy ra lỗi: {e}")
        return None
        
    finally:
        driver.quit()
        print("Đã đóng trình duyệt.")


In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
import re

def extract_lazada_data(html_file_path, category):
    print(f"Đang đọc file HTML: {html_file_path}")
    
    # Mở và đọc file HTML (nhớ dùng utf-8 nếu không sẽ bị lỗi font tiếng Việt)
    with open(html_file_path, "r", encoding="utf-8") as file:
        html_content = file.read()
        
    # Tạo đối tượng BeautifulSoup để parse HTML
    soup = BeautifulSoup(html_content, "html.parser")
    
    # Tìm TẤT CẢ các thẻ div chứa thông tin của một sản phẩm
    # Lazada dùng thuộc tính data-qa-locator="product-item" để đánh dấu
    product_items = soup.find_all(attrs={"data-qa-locator": "product-item"})
    print(f"✅ Tìm thấy {len(product_items)} sản phẩm trong trang!")
    
    parsed_data = []
    
    for item in product_items:
        # 1. Tên, Link và ID
        a_tag = item.find("a", title=True)
        title = a_tag["title"] if a_tag else None
        
        link = a_tag["href"] if a_tag and "href" in a_tag.attrs else None
        product_id = None
        
        if link:
            if link.startswith("//"):
                link = "https:" + link
            # Dùng Regex để bắt một dãy số đứng sau chữ 'pdp-i' trong link
            id_match = re.search(r'pdp-i(\d+)\.html', link)
            if id_match:
                product_id = id_match.group(1)

            
        # 2. Giá sản phẩm
        price_tag = item.find("span", class_="ooOxS")
        price = price_tag.text.strip() if price_tag else None
        
        # 3. Giảm giá (VD: 50% Off)
        discount_tag = item.find("span", class_="IcOsH")
        discount = discount_tag.text.strip() if discount_tag else None
        
        # 4. Đã bán
        sold_tag = item.find("span", class_="_1cEkb")
        sold = sold_tag.text.strip() if sold_tag else None
        
        # 5. Số lượt đánh giá
        rating_tag = item.find("span", class_="qzqFw")
        rating_count = rating_tag.text.strip().replace("(", "").replace(")", "") if rating_tag else None
        
        
        rating_score = None

        # 6. Địa điểm bán (Hà Nội, Nước ngoài...)
        location_tag = item.find("span", class_="oa6ri")
        location = location_tag.text.strip() if location_tag else None
        
        # 7. Hình ảnh
        img_tag = item.find("img")
        image_url = img_tag["src"] if img_tag and "src" in img_tag.attrs else None

        # 8. Tên shop
        shop_name = None
        
        # 8. Shop URL
        shop_url = None
        
        parsed_data.append({
            "ID Sản phẩm": product_id,
            "Tên sản phẩm": title,
            "Danh mục": category,
            "Giá": price,
            "Giảm giá": discount,
            "Đã bán": sold,
            "Điểm đánh giá": rating_score, 
            "Lượt đánh giá": rating_count,
            "Địa điểm": location,
            "Đường dẫn": link,
            "Hình ảnh": image_url,
            "Tên shop": shop_name,
            "Shop URL": shop_url
        })
        
    return parsed_data




In [ ]:
# ===================== CHẠY THỬ / CRAWL NHIỀU TRANG =====================
import os
import pandas as pd # Đảm bảo đã import pandas

BASE_URL = "https://www.lazada.vn/catalog/?from=hp_categories&page={page}&q=Trang%20ph%E1%BB%A5c%20N%E1%BB%AF&service=all_channel&src=all_channel"
TOTAL_PAGES = 1

# --- THÊM PHẦN TẠO THƯ MỤC ---
HTML_DIR = "lazada_tpnu_html_pages" # Tên thư mục bạn muốn lưu file HTML
if not os.path.exists(HTML_DIR):
    os.makedirs(HTML_DIR)
# -----------------------------

all_data = []

for page in range(1, TOTAL_PAGES + 1):
    url = BASE_URL.format(page=page)
    
    # --- CẬP NHẬT ĐƯỜNG DẪN LƯU FILE VÀO THƯ MỤC ---
    html_file = os.path.join(HTML_DIR, f"lazada_page_{page}.html")
    # ----------------------------------------------
    
    print(f"\n[{page}/{TOTAL_PAGES}] Đang crawl trang {page}...")
    html_data = check_lazada_html(url, html_file)
    
    if html_data:
        if "product-item" in html_data or "Bm3ON" in html_data:
            print("🟢 CÓ dữ liệu sản phẩm!")
        else:
            print("🔴 KHÔNG tìm thấy sản phẩm — có thể bị chặn hoặc hết trang.")
        
        data = extract_lazada_data(html_file, "Trang phục nữ")
        all_data.extend(data)
        print(f"→ Trang này: {len(data)} sản phẩm | Tổng cộng: {len(all_data)}")
    else:
        print(f"Không lấy được HTML trang {page}, bỏ qua.")

# Tổng hợp và hiển thị
print("\n" + "=" * 50)
print("BƯỚC 3: TỔNG HỢP")
print("=" * 50)

df_raw = pd.DataFrame(all_data)
print(f"Tổng trước dedup : {len(df_raw)}")
print(f"ID bị null       : {df_raw['ID Sản phẩm'].isna().sum()}")

dupes = df_raw[df_raw.duplicated(subset=["ID Sản phẩm"], keep=False)]
print(f"Số dòng trùng ID : {len(dupes)}")

df = df_raw.drop_duplicates(subset=["ID Sản phẩm"], keep="first").reset_index(drop=True)
print(f"\nHOÀN THÀNH! Tổng sản phẩm duy nhất: {len(df)}")

display(df.head())

DATA_DIR = "data"
if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)

# Nối đường dẫn lưu file
csv_path = os.path.join(DATA_DIR, "lazada_trang_phuc_nu.csv")
# -------------------------------------

# Lưu file
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"Đã lưu thành công file CSV tại: {csv_path}!")


In [ ]:
import time
import re 
from bs4 import BeautifulSoup
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from concurrent.futures import ThreadPoolExecutor
import threading
import queue
import pandas as pd
import random
import winsound  # 🔊 Thêm thư viện phát âm thanh trên Windows
import os

MAX_WORKERS = 3  # Số Chrome chạy song song

# ===================== SỬA LỖI ĐA LUỒNG =====================
# 1. Hàng đợi để phân phát ID Profile cho các luồng (Để chia làm 3 thư mục profile khác nhau)
profile_queue = queue.Queue()
for i in range(MAX_WORKERS):
    profile_queue.put(i)  # Thêm ID: 0, 1, 2 vào hàng đợi

# 2. Khóa (Lock) để tránh các luồng đụng độ nhau khi tạo file chromedriver
chrome_init_lock = threading.Lock()

def crawl_lazada_product(url):
    # Lấy 1 ID từ hàng đợi rảnh rỗi (Ngăn không cho luồng khác xài)
    profile_id = profile_queue.get()
    extracted_data = {}
    driver = None
    
    try:
        print(f"[Luồng {profile_id}] 🔄 Đang khởi tạo trình duyệt Chrome...")
        options = uc.ChromeOptions()
        options.add_argument('--disable-gpu')
        options.add_argument('--window-size=1920,1080')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        
        # CHIẾN THUẬT QUAN TRỌNG: MỖI LUỒNG DÙNG 1 FOLDER PROFILE KHÁC NHAU
        profile_path = rf'C:\Users\ADMIN\ChromeProfileData_{profile_id}'
        options.add_argument(f'--user-data-dir={profile_path}')
        
        #DÙNG KHÓA KHI KHỞI TẠO ĐỂ TRÁNH LỖI WINERROR 183
        with chrome_init_lock:
            driver = uc.Chrome(options=options)
            
        print(f"[Luồng {profile_id}] đang truy cập URL: {url}")
        driver.get(url)
        
        # --- CƠ CHẾ KIỂM TRA & VƯỢT BOT DETECTION ---
        time.sleep(3)
        if "we need to check if you are a robot" in driver.page_source.lower() or "recaptcha" in driver.page_source.lower() or "bảo vệ bởi ddos" in driver.page_source.lower():
            print(f"[Luồng {profile_id}] ⚠️ PHÁT HIỆN BOT! Vui lòng tự giải Captcha trên cửa sổ {profile_id}...")
            
            # ========================================================
            # Phát âm thanh cảnh báo bíp bíp liên tục (5 tiếng)
            try:
                for _ in range(5):
                    # winound.Beep(tần số_Hz, thời_gian_kêu_ms)
                    winsound.Beep(1500, 500) 
                    time.sleep(0.1)
            except:
                pass
            # ========================================================

            try:
                WebDriverWait(driver, 60).until(
                    EC.presence_of_element_located((By.CLASS_NAME, 'pdp-mod-review'))
                )
                print(f"[Luồng {profile_id}] ✅ Đã qua bước xác minh.")
            except Exception:
                print(f"[Luồng {profile_id}] ❌ Quá 60s không giải được.")
        
        # Chờ load trang
        try:
            WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.CLASS_NAME, 'pdp-mod-review')))
            driver.execute_script("window.scrollBy(0, 800);")
            time.sleep(4)
            driver.execute_script("window.scrollBy(0, 1000);")
            time.sleep(2)
        except Exception:
            pass
        
        # Trích xuất HTML
        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')
        
        # 1. TÊN SHOP VÀ LINK SHOP
        seller_element = soup.select_one('.seller-name-v2__detail-name, .seller-name-v2__top a, .seller-name__detail')
        if seller_element:
            extracted_data['Shop Name'] = seller_element.get_text(strip=True)
            link = seller_element.get('href')
            if not link and seller_element.find_parent('a'):
                link = seller_element.find_parent('a').get('href')
            extracted_data['Shop URL'] = "https:" + link if link and link.startswith('//') else (link or "")
        else:
            extracted_data['Shop Name'] = ""
            extracted_data['Shop URL'] = ""
            
        # 2. ĐÁNH GIÁ TRUNG BÌNH
        score_tag = soup.select_one('.container-star-v2-score, .score-average, .score, .rating-score, .pdp-review-summary__stars')
        if score_tag:
            extracted_data['Rating Score'] = score_tag.get_text(strip=True)
        else:
            all_text = ' '.join(soup.stripped_strings)
            score_regex = re.search(r'(\d\.\d)\s*\(\d+\)', all_text)
            extracted_data['Rating Score'] = score_regex.group(1) if score_regex else "" # Để trống
            
    except Exception as e:
        print(f"[Luồng {profile_id}] ❌ Lỗi khi thao tác với Chrome: {e}")
    finally:
        # Quan trọng 1: Phải luôn tắt driver kể cả khi có lỗi
        try:
            if driver:
                driver.quit()
        except:
            pass
        # Quan trọng 2: Hoàn trả lại quyền dùng profile_id vào hàng đợi cho luồng sau dùng tiếp
        profile_queue.put(profile_id)
        
    return extracted_data


# ===================== QUẢN LÝ CHẠY ĐA LUỒNG =====================
# (Giả định bạn đã có biến df chứa số liệu ở các Cell trên)
df_missing = df[df['Điểm đánh giá'].isna() | df['Shop URL'].isna()].copy()
print(f"→ Sẽ crawl {len(df_missing)} sản phẩm với {MAX_WORKERS} luồng song song...")

def crawl_one(args):
    idx, row = args
    url = row["Đường dẫn"]
    
    if pd.isna(url):
        return idx, None
    
    # Random delay này vẫn giữ để giảm tải khi mới vào, kết hợp với "Lock" ở trên là an toàn tuyệt đối 100%
    time.sleep(random.uniform(1, 4)) 
    
    detail = crawl_lazada_product(url)
    if detail:
        print(f"✅ [{idx}] {row['Tên sản phẩm'][:30]} | Rating: {detail.get('Rating Score') or 'Rỗng'} | Shop: {detail.get('Shop Name') or 'Rỗng'}")
    return idx, detail

# Chạy đa luồng
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = executor.map(crawl_one, df_missing.iterrows())
    
    for idx, detail in futures:
        if detail is None or not detail:
            continue
        # Dùng 'in detail' để lưu cả những thành phần "" vào file excel cuối
        if "Rating Score" in detail:
            df.at[idx, "Điểm đánh giá"] = detail["Rating Score"]
        if "Shop Name" in detail:
            df.at[idx, "Tên shop"] = detail["Shop Name"]
        if "Shop URL" in detail:
            df.at[idx, "Shop URL"] = detail["Shop URL"]

# Đường dẫn lưu file
csv_path = os.path.join("data", "lazada_trang_phuc_nu.csv")
# Lưu file (Đảm bảo tên file bạn muốn)
df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print(f"\n{'='*50}")
print(f"HOÀN THÀNH CÀO DỮ LIỆU!")
print(f"Rating đã có : {df['Điểm đánh giá'].notna().sum()}/{len(df)}")
print(f"Shop URL đã có: {df['Shop URL'].notna().sum()}/{len(df)}")
print(f"{'='*50}")
display(df.head())


In [ ]:
import time
import re
from bs4 import BeautifulSoup
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from concurrent.futures import ThreadPoolExecutor
import threading
import queue
import pandas as pd
import random
import winsound
import os

MAX_WORKERS = 3  # Số luồng (cửa sổ Chrome) chạy song song

# ===================== KHỞI TẠO CƠ CHẾ ĐA LUỒNG =====================
# 1. Hàng đợi để phân phát ID Profile cho các luồng (0, 1, 2)
profile_queue = queue.Queue()
for i in range(MAX_WORKERS):
    profile_queue.put(i)

# 2. Khóa (Lock) để tránh các luồng đụng độ nhau lúc khởi tạo driver
chrome_init_lock = threading.Lock()

def crawl_and_parse_shop_profile(url):
    # Lấy 1 ID từ hàng đợi rảnh rỗi
    profile_id = profile_queue.get()
    
    # Template chứa dữ liệu mặc định (để nếu lỗi vẫn không bị thiếu cột)
    extracted_data = {
        'Shop URL': url,
        'Seller ID': None,
        'Loại cửa hàng': None,
        'Followers': None,
        'Đánh giá người bán': None,
        'Danh mục chính': None,
        'Đã tham gia': None,
        'Giao đúng hẹn': None,
        'Tỷ lệ hủy đơn': None,
        'Tỷ lệ phản hồi chat': None
    }
    
    driver = None
    html = "" # Biến chứa HTML lấy được
    
    try:
        print(f"[Luồng {profile_id}] Đang khởi tạo trình duyệt Chrome...")
        options = uc.ChromeOptions()
        options.add_argument('--disable-gpu')
        options.add_argument('--window-size=1920,1080')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        
        # 🟢 CHIẾN THUẬT QUAN TRỌNG CHỐNG "BÓP" HIỆU NĂNG KHI CHẠY ĐA LUỒNG
        # Ép Chrome vẫn phải tải Javascript ở mức tối đa dù cửa sổ bị ẩn/che khuất
        options.add_argument("--disable-background-timer-throttling")
        options.add_argument("--disable-backgrounding-occluded-windows")
        options.add_argument("--disable-renderer-backgrounding")
        
        # Mỗi luồng sử dụng một thư mục user-data riêng biệt
        profile_path = rf'C:\Users\ADMIN\ChromeProfileData_{profile_id}'
        options.add_argument(f'--user-data-dir={profile_path}')
        
        # Có Lock để tránh bị lỗi WinError 183
        with chrome_init_lock:
            driver = uc.Chrome(options=options)
            
        print(f"[Luồng {profile_id}] Đang truy cập URL Hồ Sơ Cửa Hàng: \n{url}")
        driver.get(url)
        
        # ==========================================================
        # --- CƠ CHẾ KIỂM TRA & VƯỢT BOT DETECTION ---
        time.sleep(3)
        page_source_lower = driver.page_source.lower()
        if "we need to check if you are a robot" in page_source_lower or "recaptcha" in page_source_lower or "bảo vệ bởi ddos" in page_source_lower:
            print(f"[Luồng {profile_id}] PHÁT HIỆN BOT! Vui lòng tự giải Captcha trên cửa sổ {profile_id}...")
            
            try:
                for _ in range(5):
                    winsound.Beep(1500, 500)
                    time.sleep(0.1)
            except:
                pass

            try:
                WebDriverWait(driver, 60).until(
                    EC.presence_of_element_located((By.CLASS_NAME, 'seller-name__detail')) 
                )
                print(f"[Luồng {profile_id}] Đã qua bước xác minh.")
            except Exception:
                print(f"[Luồng {profile_id}] Quá 60s không giải được vòng lặp.")
        # ==========================================================
        
        print(f"[Luồng {profile_id}] Đang đợi toàn bộ chỉ số uy tín tải về màn hình...")
        
        # Đợi cơ bản 10 giây để nạp khung hình
        time.sleep(10) 
        driver.execute_script("window.scrollBy(0, 500);")
        time.sleep(5)
        
        # 🟢 KIỂM TRA LINH HOẠT XEM THÔNG TIN ĐÃ LOAD LÊN DOM CHƯA
        # Cho phép kịch bản tối đa 3 lần đợi thêm. Nếu CPU lag khiến mạng rớt nhịp, máy sẽ thông minh xin chờ thêm 5 giây.
        for _ in range(3):
            temp_source = driver.page_source
            # Kiểm tra xem có 1 trong các chữ số thống kê được render thành công chưa
            if ("Giao đúng hẹn" in temp_source) or ("Shipped on time" in temp_source) or ("Đánh giá người bán" in temp_source) or ("Đã tham gia" in temp_source):
                break
            else:
                print(f"[Luồng {profile_id}] ⏳ Web bị lag chưa kịp đổ dữ liệu xuống, đang cố đợi thêm 5s...")
                time.sleep(5)
                driver.execute_script("window.scrollBy(0, 300);")
        
        # Chỉ lưu biến HTML trực tiếp lên RAM
        html = driver.page_source
        
    except Exception as e:
        print(f"[Luồng {profile_id}] Lỗi trong quá trình thao tác với Chrome: {e}")
    finally:
        # Luôn phải tắt driver để giải phóng RAM + trả profile lại hàng đợi
        try:
            if driver:
                driver.quit()
        except:
            pass
        profile_queue.put(profile_id)
        
    # --- PHÂN TÍCH DATA NGAY TỨC THÌ NẾU LẤY ĐƯỢC HTML ---
    if html:
        print(f"[Luồng {profile_id}] Đang bóc tách dữ liệu từ HTML...")
        soup = BeautifulSoup(html, 'html.parser')
        all_text = ' '.join(soup.stripped_strings)

        is_mall = bool(soup.find('img', alt='storeIcon'))
        extracted_data['Loại cửa hàng'] = 'LazMall' if is_mall else 'Shop Thường'

        seller_id_match = re.search(r'"sellerId"\s*:\s*(\d+)', html)
        if seller_id_match:
            extracted_data['Seller ID'] = seller_id_match.group(1)
        else: 
            shop_id_match = re.search(r'"shopId"\s*:\s*(\d+)', html)
            if shop_id_match:
                extracted_data['Seller ID'] = shop_id_match.group(1)

        followers = re.search(r'([0-9,]+)\s+Followers', all_text)
        if followers: extracted_data['Followers'] = followers.group(1)

        positive_rating = re.search(r'([0-9,.]+%)\s+(?:Đánh giá người bán|Seller Ratings)', all_text)
        if positive_rating: extracted_data['Đánh giá người bán'] = positive_rating.group(1)

        category_match = re.search(r'(?:Danh mục chính|Main category)(.*?)(?:Đã tham gia|Joined)', all_text)
        if category_match: 
            clean_category = re.sub(r'[^\w\s\'\-\&]', '', category_match.group(1)).strip()
            extracted_data['Danh mục chính'] = clean_category

        joined = re.search(r'(?:Đã tham gia|Joined)(?:.*?)\s+(\d+\s*\+?\s*(?:năm|years?|tháng|months?))', all_text)
        if joined: extracted_data['Đã tham gia'] = joined.group(1).strip()

        ship_on_time = re.search(r'(?:Giao đúng hẹn|Shipped on time)(?:.*?)\s+([0-9,.]+%|Không đủ dữ liệu|Not enough data)', all_text)
        if ship_on_time: extracted_data['Giao đúng hẹn'] = ship_on_time.group(1).strip()

        cancel_rate = re.search(r'(?:Tỷ lệ hủy đơn|Cancellation rate)(?:.*?)\s+([0-9,.]+%|Không đủ dữ liệu|Not enough data)', all_text)
        if cancel_rate: extracted_data['Tỷ lệ hủy đơn'] = cancel_rate.group(1).strip()

        chat_response = re.search(r'(?:tin nhắn trong 10|10 Minutes Response Rate)(?:.*?)\s+([0-9,.]+%)', all_text)
        if chat_response: extracted_data['Tỷ lệ phản hồi chat'] = chat_response.group(1).strip()
        
        print(f"[Luồng {profile_id}] ✅ Đã trích xuất xong dữ liệu cho một shop.")
        
    return extracted_data

raw_urls = df['Shop URL'].dropna().unique()

danh_sach_shop_urls = []
for url in raw_urls:
    # Link cũ thường có dạng: https://www.lazada.vn/shop/abc-store/?itemId=123...
    # split('?') sẽ cắt lấy phần gốc: https://www.lazada.vn/shop/abc-store/
    base_url = url.split('?')[0] 
    
    # Ghép thêm đuôi đi thẳng vào Tab Profile của shop đó
    profile_url = f"{base_url}?path=profile.htm&pageTypeId=3"
    
    # Tránh trường hợp 1 shop có nhiều link kiểu base_url giống nhau nhưng parameter # nhau mọc ra
    if profile_url not in danh_sach_shop_urls:
        danh_sach_shop_urls.append(profile_url)

print(f"Tổng số Shop cần cào thông tin hồ sơ (đã lọc trùng): {len(danh_sach_shop_urls)}")


# ===================== THỰC THI CHẠY ĐA LUỒNG =====================

def wrapper_process(url):
    time.sleep(random.uniform(1, 4)) # Delay nhẹ cho đỡ gửi dồn dập
    return crawl_and_parse_shop_profile(url)

print(f"→ Sẽ crawl {len(danh_sach_shop_urls)} Shop URL với {MAX_WORKERS} luồng song song...\n")

all_results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = executor.map(wrapper_process, danh_sach_shop_urls)
    
    for result in futures:
        if result:
            all_results.append(result)

# ===== TỔNG HỢP VÀ GHI RA FILE CSV =====
df_profile = pd.DataFrame(all_results)
# Đổi tên biến sang csv_file và để đuôi .csv
csv_file = os.path.join("data", "lazada_shop_profiles_final.csv")
# Sử dụng hàm df_profile.to_csv thay vì to_excel
# Thêm utf-8-sig để đọc tiếng Việt chuẩn xác trong Excel
df_profile.to_csv(csv_file, index=False, encoding="utf-8-sig")
print(f"Đã lưu thành công file CSV tại: {csv_file}!")

print(f"\n====== HOÀN TẤT ======")
print(f"✅ Đã xử lý tổng cộng {len(all_results)} shop.")
print(f"📁 Dữ liệu được lưu thẳng vào file Excel: {excel_file}")

# Hiển thị xem trước
display(df_profile.head())


